# 4. No CAPS: quem chega e o que aparece depois (Figura 8 e Tabela 2)

Reproduz as seções **"No CAPS, gente nova chegando, não diagnóstico trocado"** e **"No CAPS, o que aparece depois do
diagnóstico"** do site. Usa as tabelas por pessoa (código pseudonimizado), que permitem seguir a mesma pessoa no
tempo dentro do CAPS. Só saem daqui totais, com contagens abaixo de 5 suprimidas.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # para importar src/ (o notebook roda de dentro de notebooks/)

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt

from src.config import *
from src.util import *

## Figura 8: pessoas que receberam F63.0 pela primeira vez, por ano

Para cada pessoa, o primeiro mês com F63.0. Quem não tinha nenhum atendimento no CAPS nos 12 meses anteriores conta
como "nova no CAPS". "Nova no CAPS" não quer dizer caso novo de vício: a pessoa pode ter sido tratada antes na rede
privada.

In [ ]:
reg = pl.read_parquet(RAW / "caps" / "sia_f63_registros.parquet")
reg = reg.filter(pl.col("cmp").is_between(pl.lit(INICIO), pl.lit(FIM_CAPS)))
reg = reg.with_columns(
    cid=pl.when(pl.col("CIDPRI").str.starts_with("F63")).then(pl.col("CIDPRI")).otherwise(pl.col("CIDASSOC")),
    data=pl.col("cmp").str.to_date("%Y%m"))
historico = pl.read_parquet(RAW / "caps" / "sia_f63_historico.parquet").with_columns(data=pl.col("cmp").str.to_date("%Y%m"))

t0 = reg.filter(pl.col("cid") == "F630").group_by("id").agg(t0=pl.col("data").min())
antes = historico.join(t0, on="id").filter(
    (pl.col("data") < pl.col("t0")) & (pl.col("data") >= pl.col("t0").dt.offset_by("-12mo")))
ja_estava = antes.select("id").unique().with_columns(ja_estava=pl.lit(True))

novos = (t0.filter(pl.col("t0") >= pl.date(2019, 1, 1))
         .join(ja_estava, on="id", how="left")
         .with_columns(pl.col("ja_estava").fill_null(False), ano=pl.col("t0").dt.year())
         .group_by("ano").agg(novas=(~pl.col("ja_estava")).sum(), ja_estavam=pl.col("ja_estava").sum())
         .sort("ano").with_columns(pct_novas=100 * pl.col("novas") / (pl.col("novas") + pl.col("ja_estavam"))))
novos

In [ ]:
fig, ax = plt.subplots(figsize=(LARGURA, 7 * CM))
anos = novos["ano"].to_list()
ax.bar(anos, novos["novas"], color=COR["caps"], label="Novas no CAPS")
ax.bar(anos, novos["ja_estavam"], bottom=novos["novas"], color=COR["caps_claro"], label="Já estavam no CAPS com outro diagnóstico")
for r in novos.to_dicts():
    ax.text(r["ano"], r["novas"] + r["ja_estavam"], f"{br(r['pct_novas'], 0)}%", ha="center", va="bottom", fontsize=7)
ax.set_xticks(anos, [str(a) if a < int(FIM_CAPS[:4]) else f"{a}*" for a in anos])
ax.set_ylabel("Pessoas")
ax.yaxis.set_major_formatter(eixo_br)
ax.legend(loc="upper left")
salvar_figura(fig, "figura8")

## Tabela 2: diagnóstico novo nos 12 meses seguintes, por motivo de entrada no CAPS

- Tabela `sia_trajetorias`: uma linha por pessoa × mês × grupo de diagnóstico, para todos os pacientes de saúde mental.
- Para cada pessoa e grupo, o primeiro mês do grupo é a entrada (t0). Entram t0 de jan/2019 a jun/2025, para haver
  12 meses de histórico antes e 12 de seguimento depois.
- Um diagnóstico é novo se aparece nos 12 meses seguintes e não aparecia nos 12 anteriores.
- "Ainda no CAPS 6–12 meses depois": algum registro do 7º ao 12º mês depois de t0.
- As linhas dos outros grupos são recalculadas como se tivessem a mesma composição por sexo e idade dos pacientes de
  jogo patológico (padronização direta).

In [ ]:
CORTE = mes_idx(FIM_CAPS)
ENTRADAS = {1: "Jogo patológico", 6: "Ansiedade", 5: "Depressão", 2: "Álcool", 3: "Drogas"}
DESFECHOS = {6: "Ansiedade", 5: "Depressão", 3: "Drogas", 2: "Álcool"}

traj = pl.read_parquet(RAW / "caps" / "sia_trajetorias.parquet").filter(pl.col("mes").is_between(0, CORTE))

# entrada: primeiro mês de cada pessoa em cada grupo, com sexo e faixa naquele mês
primeiro = traj.group_by("id", "grupo").agg(t0=pl.col("mes").min())
perfil_t0 = (traj.join(primeiro, left_on=["id", "grupo", "mes"], right_on=["id", "grupo", "t0"])
             .group_by("id", "grupo").agg(pl.col("sexo").first(), pl.col("faixa").first()))
coorte = (primeiro.filter(pl.col("grupo").is_in(list(ENTRADAS)) & pl.col("t0").is_between(12, CORTE - 12))
          .join(perfil_t0, on=["id", "grupo"]).drop_nulls(["sexo", "faixa"]).rename({"grupo": "entrada"}))

# o que a pessoa tinha nos 12 meses antes e nos 12 depois da entrada
j = traj.select("id", "mes", "grupo").join(coorte.select("id", "entrada", "t0"), on="id")
antes = j.filter(pl.col("mes").is_between(pl.col("t0") - 12, pl.col("t0") - 1)).select("id", "entrada", "grupo").unique()
depois = j.filter(pl.col("mes").is_between(pl.col("t0") + 1, pl.col("t0") + 12)).select("id", "entrada", "grupo").unique()
novo = depois.join(antes, on=["id", "entrada", "grupo"], how="anti").filter(pl.col("grupo") != pl.col("entrada"))
fica = (j.filter(pl.col("mes").is_between(pl.col("t0") + 7, pl.col("t0") + 12)).select("id", "entrada").unique()
        .with_columns(fica=pl.lit(True)))

base = coorte.join(fica, on=["id", "entrada"], how="left").with_columns(pl.col("fica").fill_null(False))
for g, nome in DESFECHOS.items():
    ids = novo.filter(pl.col("grupo") == g).select("id", "entrada").with_columns(pl.lit(True).alias(nome))
    base = base.join(ids, on=["id", "entrada"], how="left").with_columns(pl.col(nome).fill_null(False))
del traj, j, antes, depois, novo
COLS = [*DESFECHOS.values(), "fica"]

In [ ]:
# padronização direta: pesos = composição sexo × faixa dos pacientes de jogo
pesos = (base.filter(pl.col("entrada") == 1).group_by("sexo", "faixa").agg(nj=pl.len())
         .with_columns(peso=pl.col("nj") / pl.col("nj").sum()))
estratos = base.group_by("entrada", "sexo", "faixa").agg(*[pl.col(c).mean() for c in COLS])

linhas = []
for e, rotulo in ENTRADAS.items():
    pessoas = base.filter(pl.col("entrada") == e).height
    if e == 1:
        taxas = {c: 100 * base.filter(pl.col("entrada") == 1)[c].mean() for c in COLS}
    else:
        s = estratos.filter(pl.col("entrada") == e).join(pesos, on=["sexo", "faixa"])
        taxas = {c: 100 * float((s[c] * s["peso"]).sum() / s["peso"].sum()) for c in COLS}
    linhas.append({"Entrou no CAPS por": rotulo, "Pessoas": pessoas, **taxas})
tabela = pl.DataFrame(linhas)
tabela.with_columns(pl.col(COLS).round(1))

In [ ]:
def celula(r, c):
    mesmo = c != "fica" and r["Entrou no CAPS por"] == c  # o próprio diagnóstico de entrada não conta
    return "–" if mesmo else br(r[c], 1)

tabela_tex(
    "tabela2",
    "Diagnóstico novo nos 12 meses seguintes, por motivo de entrada no CAPS (%).",
    ["Entrou no CAPS por", "Pessoas", "Ansiedade (%)", "Depressão (%)", "Drogas (%)", "Álcool (%)",
     "Ainda no CAPS 6–12 meses depois (%)"],
    [[r["Entrou no CAPS por"], br(r["Pessoas"], 0), *[celula(r, c) for c in COLS]] for r in tabela.to_dicts()],
    "Entrada de jan/2019 a jun/2025. As linhas dos outros grupos estão padronizadas para a composição por sexo e idade "
    "dos pacientes de jogo patológico. Ansiedade inclui estresse (F40–F48); depressão inclui os outros transtornos do "
    "humor (F30–F39). Só vemos o que é registrado no CAPS: quem sai do CAPS some dos dados, então as taxas são um piso. "
    "Fonte: SIA-SUS (RAAS).",
    "p{3.0cm}R{1.7cm}R{1.8cm}R{1.8cm}R{1.6cm}R{1.6cm}R{2.6cm}")

jogo = tabela.row(0, named=True)
num = {"novas_2019": br(novos.filter(pl.col("ano") == 2019)["pct_novas"][0], 0),
       "novas_2025": br(novos.filter(pl.col("ano") == 2025)["pct_novas"][0], 0),
       "traj_jogo_pessoas": br(jogo["Pessoas"], 0),
       **{f"traj_jogo_{c.lower()}": br(jogo[c], 1) for c in COLS}}
num  # números citados no texto